In [1]:
!pip install implicit polars scipy numpy pandas nltk gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 70.4 MB/s eta 0:00:00


In [2]:
# -*- coding: utf-8 -*-
"""
Hyperparameter Tuning Grid Search for TF-IDF Content-Based Recommender.
Caches stemming and tokenization to run a 24-combination grid search in a few minutes.
Generates cleaned_meta.parquet automatically if it does not exist.
Saves the best parameters and metrics to a results file.
"""

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import sys
import re
import time
import gc
import html
import itertools
import joblib
import polars as pl
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer
import implicit.evaluation

# =====================================================================
# Configuration Settings for Improvements (Rocchio & Prior Boosting)
# =====================================================================
ENABLE_ROCCHIO_CENTERING = True   # Centering ratings around 3.0 (neutral) so negative reviews subtract from profile
POPULARITY_BOOST_WEIGHT = 0.05    # Additive weight for normalized log(1 + rating_number) (0.0 to disable)
QUALITY_BOOST_WEIGHT = 0.05       # Additive weight for average rating quality (0.0 to disable)



# =====================================================================
# 1. Environment & Path Configuration (Auto Colab Detection)
# =====================================================================

# Check if running locally in workspace or on Kaggle
if os.path.exists('meta.parquet'):
    BASE_DIR = '.'
    OUTPUT_DIR = './working'
elif os.path.exists('/kaggle/input/notebooks/agungramananda/preprocess-recsys/meta.parquet'):
    BASE_DIR = '/kaggle/input/notebooks/agungramananda/preprocess-recsys'
    OUTPUT_DIR = '/kaggle/working'
else:
    BASE_DIR = '.'
    OUTPUT_DIR = './working'

DATA_PATH = os.path.join(BASE_DIR, 'meta.parquet')
REVIEWS_PATH = os.path.join(BASE_DIR, 'reviews.csv')
OUTPUT_DIR = os.path.abspath(OUTPUT_DIR)
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Parse dry-run mode
DRY_RUN = False
if DRY_RUN:
    print("\n!!! DRY-RUN MODE ENABLED (Using a small sample of the dataset) !!!")

# =====================================================================
# 2. Text Preprocessing & Cleaning (Reuse content-based-tfidf.py pipeline)
# =====================================================================

stemmer = PorterStemmer()

def clean_text(text):
    if not text:
        return ""
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'[\u200b-\u200f\u2028-\u202f\ufeff]', '', text)
    text = re.sub(r'[\uff01-\uff5e]', lambda m: chr(ord(m.group()) - 0xfee0), text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def stemmed_tokenizer(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    stemmed_tokens = [
        stemmer.stem(token) 
        for token in tokens 
        if token not in ENGLISH_STOP_WORDS and len(token) > 1
    ]
    return stemmed_tokens

def generate_bigrams(tokens):
    """Generates unigrams and bigrams from a list of tokens."""
    res = list(tokens)
    for i in range(len(tokens) - 1):
        res.append(f"{tokens[i]} {tokens[i+1]}")
    return res

CLEANED_TEXT_PATH = os.path.join(OUTPUT_DIR, 'cleaned_meta.parquet')

# Check if cleaned metadata exists; if not, build it from the raw meta.parquet
if os.path.exists(CLEANED_TEXT_PATH):
    print(f"\nLoading existing cleaned metadata from {CLEANED_TEXT_PATH}...")
    cleaned_meta = pl.read_parquet(CLEANED_TEXT_PATH)
else:
    print(f"\nCleaned metadata not found at {CLEANED_TEXT_PATH}. Building from {DATA_PATH}...")
    raw_meta = pl.read_parquet(DATA_PATH)
    
    # Identify standard text columns that might exist in the dataset
    possible_text_cols = ['title', 'description', 'features', 'categories']
    available_cols = [c for c in possible_text_cols if c in raw_meta.columns]
    
    exprs = []
    for col in available_cols:
        # Handle list-type columns (like categories) by joining them with spaces
        if isinstance(raw_meta[col].dtype, pl.List):
            exprs.append(pl.col(col).list.join(" ").fill_null(""))
        else:
            exprs.append(pl.col(col).cast(pl.Utf8).fill_null(""))
            
    if exprs:
        raw_meta = raw_meta.with_columns(
            pl.concat_str(exprs, separator=" ").alias("raw_content")
        )
    else:
        print("WARNING: No text columns found. Proceeding with empty content.")
        raw_meta = raw_meta.with_columns(pl.lit("").alias("raw_content"))
        
    print("Applying text cleaning... (this might take a few moments)")
    cleaned_meta = raw_meta.with_columns(
        pl.col("raw_content").map_elements(clean_text, return_dtype=pl.Utf8).alias("content_text")
    ).select(['parent_asin', 'content_text'])
    
    print(f"Saving cleaned metadata to {CLEANED_TEXT_PATH}...")
    cleaned_meta.write_parquet(CLEANED_TEXT_PATH)

if DRY_RUN:
    print("DRY-RUN: Sampling catalog to 5,000 products...")
    cleaned_meta = cleaned_meta.head(5000)

all_asins = cleaned_meta["parent_asin"].to_list()
all_content_texts = cleaned_meta["content_text"].to_list()

asin_list = all_asins
asin_to_idx = {asin: idx for idx, asin in enumerate(asin_list)}

# Pre-tokenize all content texts into stemmed unigrams + bigrams once
print(f"\nPre-tokenizing and stemming corpus (size: {len(all_content_texts):,})...")
t0 = time.time()
pre_tokenized_bigrams = []
for doc in all_content_texts:
    unigrams = stemmed_tokenizer(doc)
    bigrams = generate_bigrams(unigrams)
    pre_tokenized_bigrams.append(bigrams)
print(f"Pre-tokenization completed in {time.time() - t0:.1f}s")

# =====================================================================
# 3. Load Reviews & Prepare Splits
# =====================================================================

# =====================================================================
# 3. Load Reviews & Prepare Splits (80:20 Holdout)
# =====================================================================

print(f"\nLoading reviews from {REVIEWS_PATH}...")
reviews_df = pl.read_csv(REVIEWS_PATH)

if DRY_RUN:
    reviews_df = reviews_df.filter(pl.col("parent_asin").is_in(all_asins))
    if reviews_df.shape[0] > 10000:
        reviews_df = reviews_df.head(10000)

print("Skipping K-core filtering...")
df_filtered = reviews_df

# 1. Deduplicate (No timestamp sorting needed for non-time-aware)
df_dedup = df_filtered.unique(subset=['user_id', 'parent_asin'], keep='first')

# 2. Create index mappings
user_mapping = df_dedup.select('user_id').unique().sort('user_id').with_row_index('user_idx')
item_mapping = df_dedup.select('parent_asin').unique().sort('parent_asin').with_row_index('item_idx')

df_indexed = df_dedup.join(user_mapping, on='user_id').join(item_mapping, on='parent_asin')

# 3. Shuffle the dataset randomly (seed ensures the split is reproducible across runs)
df_shuffled = df_indexed.sample(fraction=1.0, shuffle=True, seed=42)

# 4. Rank items randomly per user and calculate the 80% threshold
df_split = df_shuffled.with_columns(
    pl.int_range(1, pl.len() + 1).over('user_id').alias('rank'),
    pl.len().over('user_id').alias('total_count')
).with_columns(
    # max_horizontal ensures users with only 1 interaction keep it in the training set
    pl.max_horizontal(pl.lit(1), (pl.col('total_count') * 0.8).cast(pl.Int32)).alias('train_threshold')
)

# 5. Apply the 80:20 split
train_df = df_split.filter(pl.col('rank') <= pl.col('train_threshold')).select(['user_id', 'parent_asin', 'user_idx', 'item_idx', 'rating'])
test_df = df_split.filter(pl.col('rank') > pl.col('train_threshold')).select(['user_id', 'parent_asin', 'user_idx', 'item_idx', 'rating'])

# Filter splits to only contain products present in the current metadata catalog
cbf_items = set(asin_list)

train_df = train_df.filter(pl.col('parent_asin').is_in(list(cbf_items)))
test_df = test_df.filter(pl.col('parent_asin').is_in(list(cbf_items)))

n_users = len(user_mapping)
n_items = len(item_mapping)

def build_interaction_matrix(df, n_users, n_items):
    vals = df['rating'].cast(pl.Float32).to_numpy()
    user_indices = df['user_idx'].to_numpy()
    item_indices = df['item_idx'].to_numpy()
    return sparse.csr_matrix((vals, (user_indices, item_indices)), shape=(n_users, n_items))

def recall_at_k(model, train_matrix, test_matrix, K=10, batch_size=1000):
    eval_users = np.unique(test_matrix.tocoo().row)
    if len(eval_users) == 0:
        return 0.0
    recalls = []
    for i in range(0, len(eval_users), batch_size):
        batch_users = eval_users[i:i+batch_size]
        ids, _ = model.recommend(
            batch_users,
            train_matrix[batch_users],
            N=K,
            filter_already_liked_items=True
        )
        for idx, user in enumerate(batch_users):
            start_idx = test_matrix.indptr[user]
            end_idx = test_matrix.indptr[user + 1]
            true_items = set(test_matrix.indices[start_idx:end_idx])
            if not true_items:
                continue
            rec_items = ids[idx]
            hits = len(true_items.intersection(rec_items))
            recalls.append(hits / len(true_items))
    return np.mean(recalls) if recalls else 0.0

train_matrix = build_interaction_matrix(train_df, n_users, n_items)
test_matrix = build_interaction_matrix(test_df, n_users, n_items)

# Build user-item history mappings
train_user_data = train_df.group_by('user_id').agg([
    pl.col('parent_asin').alias('items'),
    pl.col('rating').alias('ratings')
]).to_pandas()
train_user_items = dict(zip(train_user_data['user_id'], train_user_data['items']))
train_user_ratings = dict(zip(train_user_data['user_id'], train_user_data['ratings']))

test_user_data = test_df.group_by('user_id').agg(pl.col('parent_asin').alias('items')).to_pandas()
test_user_items = dict(zip(test_user_data['user_id'], test_user_data['items']))

# =====================================================================
# 4. Profile Vectors & Recommender Definition (Vectorized)
# =====================================================================

class ContentBasedRecommender:
    def __init__(self, embeddings, asin_to_idx, item_mapping, user_mapping, train_items, train_ratings,
                 data_path=None, center_ratings=True, pop_weight=0.0, rating_weight=0.0):
        self.embeddings = embeddings
        self.asin_to_idx = asin_to_idx
        self.train_items = train_items
        self.train_ratings = train_ratings
        self.center_ratings = center_ratings
        self.pop_weight = pop_weight
        self.rating_weight = rating_weight

        self.user_idx_to_id = {row['user_idx']: row['user_id'] for row in user_mapping.iter_rows(named=True)}
        self.item_id_to_idx = {row['parent_asin']: row['item_idx'] for row in item_mapping.iter_rows(named=True)}

        n_items = len(item_mapping)
        item_idx_to_asin = [None] * n_items
        for row in item_mapping.iter_rows(named=True):
            item_idx_to_asin[row['item_idx']] = row['parent_asin']
            
        dummy_row = sparse.csr_matrix((1, embeddings.shape[1]), dtype=np.float32)
        embeddings_with_dummy = sparse.vstack([embeddings, dummy_row]).tocsr()
            
        dummy_idx = embeddings.shape[0]
        aligned_indices = []
        for item_idx in range(n_items):
            asin = item_idx_to_asin[item_idx]
            embed_idx = asin_to_idx.get(asin)
            if embed_idx is not None:
                aligned_indices.append(embed_idx)
            else:
                aligned_indices.append(dummy_idx)
                
        self.aligned_embeddings = embeddings_with_dummy[aligned_indices]

        # Load popularity and quality statistics for boosting
        if data_path and os.path.exists(data_path) and (pop_weight > 0.0 or rating_weight > 0.0):
            print("Loading item stats from meta.parquet for popularity/quality boost...")
            meta_stats = pl.read_parquet(data_path, columns=['parent_asin', 'average_rating', 'rating_number'])
            stats_df = item_mapping.join(meta_stats, on='parent_asin', how='left')
            
            self.avg_ratings = stats_df['average_rating'].fill_null(3.0).cast(pl.Float32).to_numpy()
            self.rating_numbers = stats_df['rating_number'].fill_null(0).cast(pl.Float32).to_numpy()
            
            # log(1 + rating_number) normalized to [0, 1] range
            log_pop = np.log1p(self.rating_numbers)
            max_pop = log_pop.max()
            self.log_popularity = (log_pop / max_pop) if max_pop > 0 else log_pop
        else:
            self.avg_ratings = None
            self.log_popularity = None

    def recommend(self, userid, user_items, N=10, filter_already_liked_items=True, **kwargs):
        is_scalar = np.isscalar(userid)
        users = [userid] if is_scalar else userid

        if not is_scalar and user_items is not None:
            # 1. Apply Rocchio Mean-Centering if enabled (ratings - 3.0)
            if self.center_ratings:
                user_items_proc = user_items.copy()
                user_items_proc.data = user_items_proc.data - 3.0
            else:
                user_items_proc = user_items

            P = user_items_proc.dot(self.aligned_embeddings)
            if sparse.issparse(P):
                P = P.toarray()
                
            norms = np.linalg.norm(P, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            P = P / norms

            scores = self.aligned_embeddings.dot(P.T).T
            if sparse.issparse(scores):
                scores = scores.toarray()

            # 2. Additive Quality & Popularity Boosting
            if self.log_popularity is not None and self.pop_weight > 0.0:
                scores = scores + self.pop_weight * self.log_popularity
                
            if self.avg_ratings is not None and self.rating_weight > 0.0:
                # Scale average ratings [1.0, 5.0] to [0.0, 1.0] for additive prior
                norm_ratings = (self.avg_ratings - 1.0) / 4.0
                scores = scores + self.rating_weight * norm_ratings

            if filter_already_liked_items:
                rows, cols = user_items.nonzero()
                scores[rows, cols] = -np.inf

            top_indices = np.argsort(scores, axis=1)[:, ::-1][:, :N]
            top_scores = np.take_along_axis(scores, top_indices, axis=1)

            return top_indices.astype(np.int32), top_scores.astype(np.float32)

        # Single user fallback path (scalar)
        batch_recs = []
        batch_scores = []
        for i, u in enumerate(users):
            user_id_str = self.user_idx_to_id.get(u)
            if user_id_str is None:
                batch_recs.append([0] * N)
                batch_scores.append([0.0] * N)
                continue
            
            items = self.train_items.get(user_id_str, [])
            ratings = self.train_ratings.get(user_id_str, [])
            valid_pairs = [(self.asin_to_idx[a], float(r)) for a, r in zip(items, ratings) if a in self.asin_to_idx]
            if not valid_pairs:
                batch_recs.append([0] * N)
                batch_scores.append([0.0] * N)
                continue
            
            indices = [p[0] for p in valid_pairs]
            r_ui = np.array([p[1] for p in valid_pairs], dtype=np.float32)
            
            if self.center_ratings:
                r_ui = r_ui - 3.0
                
            v_i = self.embeddings[indices]
            p_u = v_i.T.dot(r_ui) / np.sum(np.abs(r_ui))
            
            norm = np.linalg.norm(p_u)
            if norm > 0:
                p_u = p_u / norm

            scores = self.aligned_embeddings.dot(p_u)
            
            if self.log_popularity is not None and self.pop_weight > 0.0:
                scores = scores + self.pop_weight * self.log_popularity
            if self.avg_ratings is not None and self.rating_weight > 0.0:
                norm_ratings = (self.avg_ratings - 1.0) / 4.0
                scores = scores + self.rating_weight * norm_ratings

            if filter_already_liked_items and user_items is not None:
                liked_indices = user_items[i].indices
                scores[liked_indices] = -np.inf

            top_indices = np.argsort(scores)[::-1][:N]
            batch_recs.append(top_indices)
            batch_scores.append(scores[top_indices])

        if is_scalar:
            return np.array(batch_recs[0], dtype=np.int32), np.array(batch_scores[0], dtype=np.float32)
        return np.array(batch_recs, dtype=np.int32), np.array(batch_scores, dtype=np.float32)

# =====================================================================

def build_tfidf_weighted_w2v_embeddings(pre_tokenized_corpus, tfidf_matrix, tfidf_vectorizer, w2v_model, vector_size=128):
    print("Generating TF-IDF weighted Word2Vec embeddings (vectorized/optimized)...")
    t0 = time.time()
    
    feature_names = tfidf_vectorizer.get_feature_names_out()
    word_to_feature_idx = {word: idx for idx, word in enumerate(feature_names)}
    
    vocab = w2v_model.wv
    n_docs = len(pre_tokenized_corpus)
    embeddings = np.zeros((n_docs, vector_size), dtype=np.float32)
    
    indptr = tfidf_matrix.indptr
    indices = tfidf_matrix.indices
    data = tfidf_matrix.data
    
    for doc_idx in range(n_docs):
        doc = pre_tokenized_corpus[doc_idx]
        start = indptr[doc_idx]
        end = indptr[doc_idx + 1]
        
        # Build weight lookup dict for this document
        doc_weights_lookup = {indices[i]: data[i] for i in range(start, end)}
        
        doc_vectors = []
        doc_weights = []
        for word in doc:
            if word in vocab:
                vec = vocab[word]
                f_idx = word_to_feature_idx.get(word)
                weight = doc_weights_lookup.get(f_idx, 1.0) if f_idx is not None else 1.0
                doc_vectors.append(vec)
                doc_weights.append(weight)
                
        if doc_vectors:
            weighted_avg = np.average(doc_vectors, axis=0, weights=doc_weights)
            embeddings[doc_idx] = weighted_avg
            
    # L2 normalize
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms
    
    print(f"TF-IDF weighted Word2Vec embeddings generated in {time.time() - t0:.1f}s")
    return sparse.csr_matrix(embeddings)


# 5. Hyperparameter Tuning Grid Search
# =====================================================================

# Hyperparameter search space
param_grid = {
    'max_features': [15000, 25000, 35000],
    'ngram_range': [(1, 1), (1, 2)],
    'min_df': [2, 5],
    'sublinear_tf': [True, False]
}

keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"\n==================================================")
print(f"       STARTING TF-IDF GRID SEARCH")
print(f"       Total experiments to run: {len(experiments)}")
print(f"==================================================")

num_eval_users = min(5000, n_users)
eval_train = train_matrix[:num_eval_users]
eval_test = test_matrix[:num_eval_users]

results = []
best_prec = -1.0
best_params = None

for idx, params in enumerate(experiments):
    t_start = time.time()
    
    # 1. Filter the pre-tokenized corpus based on ngram_range
    # Pre-tokenized corpus has unigrams + bigrams.
    # If ngram_range is (1, 1), filter out tokens that contain space
    if params['ngram_range'] == (1, 1):
        filtered_corpus = [[t for t in doc if " " not in t] for doc in pre_tokenized_bigrams]
    else:
        filtered_corpus = pre_tokenized_bigrams
        
    # 2. Fit TF-IDF using identity analyzer (extremely fast pass-through)
    vectorizer = TfidfVectorizer(
        analyzer=lambda x: x,
        max_features=params['max_features'],
        min_df=params['min_df'],
        sublinear_tf=params['sublinear_tf'],
        norm='l2'
    )
    
    embeddings = vectorizer.fit_transform(filtered_corpus)
    
    # 3. Instantiate Recommender
    cbf_model = ContentBasedRecommender(
        embeddings, asin_to_idx, item_mapping, user_mapping,
        train_user_items, train_user_ratings,
        data_path=DATA_PATH,
        center_ratings=ENABLE_ROCCHIO_CENTERING,
        pop_weight=POPULARITY_BOOST_WEIGHT,
        rating_weight=QUALITY_BOOST_WEIGHT
    )
    
    # 4. Evaluate Precision@10, NDCG@10 & Recall@10
    p_at_10 = implicit.evaluation.precision_at_k(
        cbf_model, eval_train, eval_test, K=10, show_progress=False
    )
    n_at_10 = implicit.evaluation.ndcg_at_k(
        cbf_model, eval_train, eval_test, K=10, show_progress=False
    )
    r_at_10 = recall_at_k(
        cbf_model, eval_train, eval_test, K=10
    )
    
    elapsed = time.time() - t_start
    print(f"Exp {idx+1:02d}/{len(experiments)} | "
          f"max_feat={params['max_features']:<5} | "
          f"ngrams={str(params['ngram_range']):<6} | "
          f"min_df={params['min_df']} | "
          f"sublinear={str(params['sublinear_tf']):<5} | "
          f"Prec@10: {p_at_10:.6f} | Recall@10: {r_at_10:.6f} | NDCG@10: {n_at_10:.6f} | ({elapsed:.1f}s)")
          
    results.append({
        "max_features": params['max_features'],
        "ngram_range": str(params['ngram_range']),
        "min_df": params['min_df'],
        "sublinear_tf": params['sublinear_tf'],
        "Precision@10": p_at_10,
        "Recall@10": r_at_10,
        "NDCG@10": n_at_10,
        "Time (s)": elapsed
    })
    
    if p_at_10 > best_prec:
        best_prec = p_at_10
        best_params = params

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, 'tfidf_tuning_results.csv'), index=False)

# Find corresponding best recall and ndcg for best precision params
best_exp_row = results_df.iloc[results_df['Precision@10'].idxmax()]

# Save summary text file
summary_path = os.path.join(OUTPUT_DIR, 'tfidf_tuning_summary.txt')
with open(summary_path, 'w') as f:
    f.write("=== TF-IDF Hyperparameter Tuning Summary ===\n\n")
    f.write(f"Best Parameters Found:\n")
    for k, v in best_params.items():
        f.write(f"  - {k}: {v}\n")
    f.write(f"\nBest Precision@10: {best_prec:.6f}\n")
    f.write(f"Best Recall@10: {best_exp_row['Recall@10']:.6f}\n")
    f.write(f"Best NDCG@10: {best_exp_row['NDCG@10']:.6f}\n")
    f.write(f"Results table saved to: tfidf_tuning_results.csv\n")

print(f"\n==================================================")
print(f"Best Hyperparameters:")
for k, v in best_params.items():
    print(f"  - {k}: {v}")
print(f"Best Precision@10: {best_prec:.6f}")
print(f"Best Recall@10: {best_exp_row['Recall@10']:.6f}")
print(f"Best NDCG@10: {best_exp_row['NDCG@10']:.6f}")
print(f"Full results table saved to: {os.path.join(OUTPUT_DIR, 'tfidf_tuning_results.csv')}")
print(f"Summary report saved to: {summary_path}")
print(f"==================================================")


# =====================================================================
# 6. Comparative Experiment: Custom Word2Vec vs. TF-IDF
# =====================================================================

print("\n==================================================")
print("       RUNNING WORD2VEC EXPERIMENTS")
print("==================================================")

# 1. Train Word2Vec model from scratch on the tokenized corpus
from gensim.models import Word2Vec

print("Training Word2Vec model from scratch on corpus...")
t_w2v_start = time.time()
w2v_model = Word2Vec(
    sentences=pre_tokenized_bigrams, 
    vector_size=128, 
    window=5, 
    min_count=2, 
    workers=4, 
    epochs=10,
    seed=42
)
print(f"Word2Vec trained in {time.time() - t_w2v_start:.1f}s")

# 2. Build Average Word2Vec embeddings
vocab = w2v_model.wv
w2v_dense = np.zeros((len(pre_tokenized_bigrams), 128), dtype=np.float32)
for doc_idx, doc in enumerate(pre_tokenized_bigrams):
    valid_vecs = [vocab[w] for w in doc if w in vocab]
    if valid_vecs:
        w2v_dense[doc_idx] = np.mean(valid_vecs, axis=0)

# L2 Normalize
norms = np.linalg.norm(w2v_dense, axis=1, keepdims=True)
norms[norms == 0] = 1.0
w2v_dense = w2v_dense / norms
w2v_sparse_avg = sparse.csr_matrix(w2v_dense)

# Evaluate Average Word2Vec Recommender
w2v_avg_model = ContentBasedRecommender(
    w2v_sparse_avg, asin_to_idx, item_mapping, user_mapping,
    train_user_items, train_user_ratings,
    data_path=DATA_PATH,
    center_ratings=ENABLE_ROCCHIO_CENTERING,
    pop_weight=POPULARITY_BOOST_WEIGHT,
    rating_weight=QUALITY_BOOST_WEIGHT
)

p_w2v_avg = implicit.evaluation.precision_at_k(w2v_avg_model, eval_train, eval_test, K=10, show_progress=False)
n_w2v_avg = implicit.evaluation.ndcg_at_k(w2v_avg_model, eval_train, eval_test, K=10, show_progress=False)
r_w2v_avg = recall_at_k(w2v_avg_model, eval_train, eval_test, K=10)

print(f"Word2Vec (Simple Avg)   | Prec@10: {p_w2v_avg:.6f} | Recall@10: {r_w2v_avg:.6f} | NDCG@10: {n_w2v_avg:.6f}")

# 3. Build TF-IDF Weighted Word2Vec embeddings
# We use the best vectorizer parameters from grid search (or default to best_params if defined)
best_max_features = best_params.get('max_features', 15000)
best_ngram_range = best_params.get('ngram_range', (1, 1))
best_min_df = best_params.get('min_df', 2)
best_sublinear = best_params.get('sublinear_tf', True)

if best_ngram_range == (1, 1):
    filtered_corpus_w2v = [[t for t in doc if " " not in t] for doc in pre_tokenized_bigrams]
else:
    filtered_corpus_w2v = pre_tokenized_bigrams

best_vectorizer = TfidfVectorizer(
    analyzer=lambda x: x,
    max_features=best_max_features,
    min_df=best_min_df,
    sublinear_tf=best_sublinear,
    norm='l2'
)
best_tfidf_matrix = best_vectorizer.fit_transform(filtered_corpus_w2v)

# Generate weighted embeddings using optimized helper
w2v_weighted_sparse = build_tfidf_weighted_w2v_embeddings(
    filtered_corpus_w2v, best_tfidf_matrix, best_vectorizer, w2v_model, vector_size=128
)

# Evaluate Weighted Word2Vec Recommender
w2v_weighted_model = ContentBasedRecommender(
    w2v_weighted_sparse, asin_to_idx, item_mapping, user_mapping,
    train_user_items, train_user_ratings,
    data_path=DATA_PATH,
    center_ratings=ENABLE_ROCCHIO_CENTERING,
    pop_weight=POPULARITY_BOOST_WEIGHT,
    rating_weight=QUALITY_BOOST_WEIGHT
)

p_w2v_weighted = implicit.evaluation.precision_at_k(w2v_weighted_model, eval_train, eval_test, K=10, show_progress=False)
n_w2v_weighted = implicit.evaluation.ndcg_at_k(w2v_weighted_model, eval_train, eval_test, K=10, show_progress=False)
r_w2v_weighted = recall_at_k(w2v_weighted_model, eval_train, eval_test, K=10)

print(f"Word2Vec (Weighted Avg) | Prec@10: {p_w2v_weighted:.6f} | Recall@10: {r_w2v_weighted:.6f} | NDCG@10: {n_w2v_weighted:.6f}")

# Append to summary report
with open(summary_path, 'a') as f:
    f.write("\n=== Word2Vec Comparative Experiment Results ===\n")
    f.write(f"Word2Vec (Simple Average):\n")
    f.write(f"  - Precision@10: {p_w2v_avg:.6f}\n")
    f.write(f"  - Recall@10: {r_w2v_avg:.6f}\n")
    f.write(f"  - NDCG@10: {n_w2v_avg:.6f}\n")
    f.write(f"Word2Vec (TF-IDF Weighted Average):\n")
    f.write(f"  - Precision@10: {p_w2v_weighted:.6f}\n")
    f.write(f"  - Recall@10: {r_w2v_weighted:.6f}\n")
    f.write(f"  - NDCG@10: {n_w2v_weighted:.6f}\n")

print("\n==================================================")
print("Word2Vec comparative results saved to summary report!")
print("==================================================")



Cleaned metadata not found at /kaggle/working/cleaned_meta.parquet. Building from /kaggle/input/notebooks/agungramananda/preprocess-recsys/meta.parquet...
Applying text cleaning... (this might take a few moments)
Saving cleaned metadata to /kaggle/working/cleaned_meta.parquet...

Pre-tokenizing and stemming corpus (size: 126,215)...
Pre-tokenization completed in 173.3s

Loading reviews from /kaggle/input/notebooks/agungramananda/preprocess-recsys/reviews.csv...
Skipping K-core filtering...

       STARTING TF-IDF GRID SEARCH
       Total experiments to run: 24
Exp 01/24 | max_feat=15000 | ngrams=(1, 1) | min_df=2 | sublinear=True  | Prec@10: 0.022668 | Recall@10: 0.022773 | NDCG@10: 0.012577 | (68.5s)
Exp 02/24 | max_feat=15000 | ngrams=(1, 1) | min_df=2 | sublinear=False | Prec@10: 0.019430 | Recall@10: 0.020464 | NDCG@10: 0.010988 | (67.8s)
Exp 03/24 | max_feat=15000 | ngrams=(1, 1) | min_df=5 | sublinear=True  | Prec@10: 0.022668 | Recall@10: 0.022773 | NDCG@10: 0.012199 | (67.8s)
